# Delta Lake MERGE Assignment


In [8]:
import os
import sys
from pathlib import Path

import pandas as pd

print('Starting load step...')
print('Python executable:', sys.executable)
print('Pandas version:', pd.__version__)

base = Path.cwd()
for candidate in [base, *base.parents]:
    if (candidate / 'delta-lake-assignment' / 'data' / 'customer_master.csv').exists():
        project_root = candidate / 'delta-lake-assignment'
        break
else:
    raise FileNotFoundError('Could not find the assignment folder.')

os.chdir(project_root.parent)
print('Project root:', project_root)


def read_csv_robust(path):
    for encoding in ['utf-8', 'utf-8-sig', 'latin1']:
        try:
            return pd.read_csv(path, encoding=encoding)
        except Exception:
            continue
    return pd.read_csv(path, encoding='utf-8', errors='replace')


master_df = read_csv_robust(project_root / 'data' / 'customer_master.csv')
incremental_df = read_csv_robust(project_root / 'data' / 'customer_incremental.csv')

output_path = project_root / 'delta_output'
output_path.mkdir(exist_ok=True)
print('Output folder:', output_path)

print('Loaded rows:', len(master_df), len(incremental_df))

Starting load step...
Python executable: c:\Users\ASUS\AppData\Local\Programs\Python\Python313\python.exe
Pandas version: 2.3.2
Project root: c:\Users\ASUS\OneDrive\Desktop\Assignment_7\delta-lake-assignment
Output folder: c:\Users\ASUS\OneDrive\Desktop\Assignment_7\delta-lake-assignment\delta_output
Loaded rows: 12 11


In [9]:
master_df = master_df.drop_duplicates().reset_index(drop=True)
incremental_df = incremental_df.drop_duplicates().reset_index(drop=True)

for col in master_df.columns:
    master_df[col] = master_df[col].fillna('Unknown')
for col in incremental_df.columns:
    incremental_df[col] = incremental_df[col].fillna('Unknown')

print('After cleaning')
print(master_df.head())
print(incremental_df.head())

After cleaning
   customer_id first_name last_name     city state              email
0         1001      Alice   Johnson  Seattle    WA  alice@example.com
1         1002        Bob     Smith   Denver    CO    bob@example.com
2         1003      Carol       Lee   Austin    TX  carol@example.com
3         1004      David       Kim   Boston    MA  david@example.com
4         1005        Eve    Garcia  Chicago    IL    eve@example.com
   customer_id first_name last_name       city state                    email
0         1002        Bob     Smith     Denver    CO  bob.updated@example.com
1         1003      Carol       Lee     Austin    TX        carol@example.com
2         1006      Frank  Martinez      Miami    FL        frank@example.com
3         1007      Grace    Wilson   Portland    OR        grace@example.com
4         1008      Heena     Patel  San Diego    CA    heena.new@example.com


In [10]:
from deltalake import DeltaTable, write_deltalake

before_count = len(master_df)

delta_table_path = output_path / 'customer_master_delta'

write_deltalake(delta_table_path, master_df, mode='overwrite')

dt = DeltaTable(delta_table_path)

dt.merge(
    source=incremental_df,
    predicate='target.customer_id = source.customer_id',
    source_alias='source',
    target_alias='target'
).when_matched_update_all().when_not_matched_insert_all().execute()

history = dt.history(1)[0]
metrics = history.get('operationMetrics', {})
updated_rows = int(metrics.get('num_target_rows_updated', 0))
inserted_rows = int(metrics.get('num_target_rows_inserted', 0))

merged_df = dt.to_pandas()
merged_df = merged_df.drop_duplicates(subset=['customer_id']).reset_index(drop=True)
after_count = len(merged_df)

print('Rows before merge:', before_count)
print('Rows after merge:', after_count)
print('Updated rows:', updated_rows)
print('Inserted rows:', inserted_rows)
print('Duplicate count:', after_count - merged_df.drop_duplicates(subset=['customer_id']).shape[0])
print(merged_df.to_string(index=False))

Rows before merge: 12
Rows after merge: 19
Updated rows: 4
Inserted rows: 7
Duplicate count: 0
 customer_id first_name last_name      city state                   email
        1006      Frank  Martinez     Miami    FL       frank@example.com
        1007      Grace    Wilson  Portland    OR       grace@example.com
        1015       Omar    Hassan Las Vegas    NV        omar@example.com
        1016      Priya       Das   Houston    TX       priya@example.com
        1017      Quinn     Adams   Memphis    TN       quinn@example.com
        1018      Rahul     Mehta    Denver    CO       rahul@example.com
        1019       Sara     Ahmed Baltimore    MD        sara@example.com
        1002        Bob     Smith    Denver    CO bob.updated@example.com
        1003      Carol       Lee    Austin    TX       carol@example.com
        1008      Heena     Patel San Diego    CA   heena.new@example.com
        1009      Imran      Khan   Phoenix    AZ       imran@example.com
        1001     

In [11]:
output_file = output_path / 'merged_customers.csv'
merged_df.to_csv(output_file, index=False)

print('Saved output:', output_file)
print('Merged CSV preview:')
print(pd.read_csv(output_file).head(10).to_string(index=False))

Saved output: c:\Users\ASUS\OneDrive\Desktop\Assignment_7\delta-lake-assignment\delta_output\merged_customers.csv
Merged CSV preview:
 customer_id first_name last_name      city state                   email
        1006      Frank  Martinez     Miami    FL       frank@example.com
        1007      Grace    Wilson  Portland    OR       grace@example.com
        1015       Omar    Hassan Las Vegas    NV        omar@example.com
        1016      Priya       Das   Houston    TX       priya@example.com
        1017      Quinn     Adams   Memphis    TN       quinn@example.com
        1018      Rahul     Mehta    Denver    CO       rahul@example.com
        1019       Sara     Ahmed Baltimore    MD        sara@example.com
        1002        Bob     Smith    Denver    CO bob.updated@example.com
        1003      Carol       Lee    Austin    TX       carol@example.com
        1008      Heena     Patel San Diego    CA   heena.new@example.com


In [13]:
summary = (
    f"\nDelta Lake Merge Summary\n"
    f"Rows before merge: {before_count}\n"
    f"Rows after merge: {after_count}\n"
    f"Updated rows: {updated_rows}\n"
    f"Inserted rows: {inserted_rows}\n"
    f"Duplicates in final table: {after_count - merged_df.drop_duplicates(subset=['customer_id']).shape[0]}\n"
    f"Final table saved to: {output_file}\n"
)
print(summary)


Delta Lake Merge Summary
Rows before merge: 12
Rows after merge: 19
Updated rows: 4
Inserted rows: 7
Duplicates in final table: 0
Final table saved to: c:\Users\ASUS\OneDrive\Desktop\Assignment_7\delta-lake-assignment\delta_output\merged_customers.csv

